# CatBoost Health Condition Classifier

CatBoost can use categorical columns directly, so this notebook uses the readable feature files (`train_split_features.csv`, `val_split_features.csv`, `test_features.csv`) instead of the numeric one-hot-only files. It trains a few CatBoost variants and saves separate submissions.

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

RANDOM_STATE = 42
ID_COL = "id"
TARGET_COL = "health_condition"

## Load Feature Data

In [ ]:
train_df = pd.read_csv("data/train_split_features.csv")
val_df = pd.read_csv("data/val_split_features.csv")
test_df = pd.read_csv("data/test_features.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

feature_cols = [col for col in train_df.columns if col not in [ID_COL, TARGET_COL]]
X_train = train_df[feature_cols].copy()
y_train_raw = train_df[TARGET_COL].copy()
X_val = val_df[feature_cols].copy()
y_val_raw = val_df[TARGET_COL].copy()
X_test = test_df[feature_cols].copy()

cat_features = [col for col in feature_cols if X_train[col].dtype == "object"]
cat_feature_indices = [feature_cols.index(col) for col in cat_features]

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)
print("feature count:", len(feature_cols))
print("categorical feature count:", len(cat_features))
print("categorical features:", cat_features)

## Target Encoding For Metrics

In [ ]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
class_names = label_encoder.classes_.tolist()

label_mapping = pd.DataFrame({"class_id": range(len(class_names)), "health_condition": class_names})
label_mapping

## CatBoost Pools

In [ ]:
train_pool = Pool(X_train, y_train, cat_features=cat_feature_indices, feature_names=feature_cols)
val_pool = Pool(X_val, y_val, cat_features=cat_feature_indices, feature_names=feature_cols)
test_pool = Pool(X_test, cat_features=cat_feature_indices, feature_names=feature_cols)

## Train Variants

In [ ]:
variant_configs = [
    {"name": "plain", "auto_class_weights": None},
    {"name": "balanced", "auto_class_weights": "Balanced"},
    {"name": "sqrt_balanced", "auto_class_weights": "SqrtBalanced"},
]

validation_rows = []
trained_models = {}

for config in variant_configs:
    print("Training", config["name"], "auto_class_weights=", config["auto_class_weights"])
    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="Accuracy",
        iterations=2500,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=6.0,
        random_seed=RANDOM_STATE,
        auto_class_weights=config["auto_class_weights"],
        od_type="Iter",
        od_wait=120,
        verbose=100,
        allow_writing_files=False,
    )
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    val_pred = model.predict(val_pool).astype(int).ravel()
    val_pred_labels = label_encoder.inverse_transform(val_pred)
    dist = pd.Series(val_pred_labels).value_counts(normalize=True).mul(100).to_dict()

    validation_rows.append({
        "variant": config["name"],
        "accuracy": accuracy_score(y_val_raw, val_pred_labels),
        "macro_f1": f1_score(y_val_raw, val_pred_labels, average="macro"),
        "weighted_f1": f1_score(y_val_raw, val_pred_labels, average="weighted"),
        "best_iteration": model.get_best_iteration(),
        "val_at_risk_pct": dist.get("at-risk", 0),
        "val_fit_pct": dist.get("fit", 0),
        "val_unhealthy_pct": dist.get("unhealthy", 0),
    })
    trained_models[config["name"]] = model

validation_report = pd.DataFrame(validation_rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
validation_report

## Save Validation Reports

In [ ]:
best_variant = validation_report.iloc[0]["variant"]
best_model = trained_models[best_variant]
print("best local variant:", best_variant)
print(classification_report(y_val_raw, label_encoder.inverse_transform(best_model.predict(val_pool).astype(int).ravel()), labels=class_names))

confusion = pd.DataFrame(
    confusion_matrix(y_val_raw, label_encoder.inverse_transform(best_model.predict(val_pool).astype(int).ravel()), labels=class_names),
    index=[f"actual_{label}" for label in class_names],
    columns=[f"pred_{label}" for label in class_names],
)
confusion

## Final Full-Data Training And Submissions

Each variant is retrained on train + validation using its best iteration count, then used to predict test. This produces multiple submission files for public-score testing.

In [ ]:
full_df = pd.concat([train_df, val_df], ignore_index=True)
X_full = full_df[feature_cols].copy()
y_full = label_encoder.transform(full_df[TARGET_COL])
full_pool = Pool(X_full, y_full, cat_features=cat_feature_indices, feature_names=feature_cols)

submission_rows = []

for config in variant_configs:
    variant = config["name"]
    best_iteration = trained_models[variant].get_best_iteration()
    final_iterations = int(best_iteration + 1) if best_iteration is not None and best_iteration >= 0 else 500

    print("Final training", variant, "iterations=", final_iterations)
    final_model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="Accuracy",
        iterations=final_iterations,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=6.0,
        random_seed=RANDOM_STATE,
        auto_class_weights=config["auto_class_weights"],
        verbose=100,
        allow_writing_files=False,
    )
    final_model.fit(full_pool)

    test_pred = final_model.predict(test_pool).astype(int).ravel()
    test_labels = label_encoder.inverse_transform(test_pred)

    submission = sample_submission.copy()
    submission[ID_COL] = test_df[ID_COL].values
    submission[TARGET_COL] = test_labels

    output_path = f"data/submission_catboost_{variant}.csv"
    submission.to_csv(output_path, index=False)

    dist = submission[TARGET_COL].value_counts(normalize=True).mul(100).to_dict()
    submission_rows.append({
        "variant": variant,
        "path": output_path,
        "test_at_risk_pct": dist.get("at-risk", 0),
        "test_fit_pct": dist.get("fit", 0),
        "test_unhealthy_pct": dist.get("unhealthy", 0),
    })

submission_report = pd.DataFrame(submission_rows)
submission_report

## Combined Report

In [ ]:
combined_report = validation_report.merge(submission_report, on="variant", how="left")
combined_report